# DocFinder v2 — quickstart Colab

**6 cellules** à exécuter dans l'ordre :

1. Install deps  →  **Runtime → Restart session** (obligatoire après install)
2. Env vars + clone du repo (branche `claude/review-project-cloudflare-ggcYD`)
3. Cleanup + cloudflared tunnel #2 (`encode.jinkohub.digital` → `:8001`)
4. Préflight CF Access (fail-fast si la policy n'autorise pas le service token)
5. **Flags runtime** : `DOCFINDER_FORCE_REINDEX`, `DOCFINDER_MAX_DOCS`
6. `query_server` + pipeline d'indexation

**Prérequis** : runtime GPU T4 + secrets Colab (`MAC_BASE_URL`, `COLAB_QUERY_TOKEN`, `CF_ACCESS_CLIENT_ID`, `CF_ACCESS_CLIENT_SECRET`, `TOKEN_TUNNEL_2`).

In [ ]:
# ─── Cellule 1 : deps ───
# Après cette cellule : Runtime → Restart session (obligatoire)
!pip install -q FlagEmbedding==1.3.5 'transformers>=4.45,<4.50' 'fastapi>=0.110' 'uvicorn[standard]>=0.27' httpx pydantic nest_asyncio pymupdf python-docx python-pptx openpyxl easyocr

In [ ]:
# ─── Cellule 2 : env + clone (avec vérif HEAD) ───
import os, subprocess, shutil, sys
from google.colab import userdata

BRANCH = "claude/review-project-cloudflare-ggcYD"

os.environ["MAC_BASE_URL"]            = userdata.get("MAC_BASE_URL")
os.environ["DOCFINDER_ROOT"]          = "/Users/julien/Documents"
os.environ["COLAB_QUERY_TOKEN"]       = userdata.get("COLAB_QUERY_TOKEN")
os.environ["CF_ACCESS_CLIENT_ID"]     = userdata.get("CF_ACCESS_CLIENT_ID")
os.environ["CF_ACCESS_CLIENT_SECRET"] = userdata.get("CF_ACCESS_CLIENT_SECRET")

assert os.environ["COLAB_QUERY_TOKEN"].strip(), "COLAB_QUERY_TOKEN manquant (Colab secrets)."
assert os.environ["MAC_BASE_URL"].startswith("https://"), "MAC_BASE_URL invalide."
assert len(os.environ["CF_ACCESS_CLIENT_SECRET"]) >= 32, "CF_ACCESS_CLIENT_SECRET trop court."

# Clone frais — sortir AVANT rmtree (cwd-delete trap).
os.chdir("/content")
shutil.rmtree("/content/docfinder", ignore_errors=True)
subprocess.run(["git", "clone", "https://github.com/kofekod23/docfinder.git", "/content/docfinder"], check=True)
subprocess.run(["git", "-C", "/content/docfinder", "checkout", BRANCH], check=True)
os.chdir("/content/docfinder")

# Purge cache d'import au cas où une cellule précédente aurait importé l'ancien colab/.
for mod in list(sys.modules):
    if mod.startswith("colab"):
        del sys.modules[mod]

branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
head   = subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip()
print(f"[ok] branche          : {branch}")
print(f"[ok] HEAD             : {head}")
print(f"[ok] helpers présent  : {os.path.exists('/content/docfinder/colab_helpers_cell.py')}")
print(f"[ok] CF_ACCESS_ID     : {os.environ['CF_ACCESS_CLIENT_ID'][:12]}…")
print(f"[ok] MAC_BASE_URL     : {os.environ['MAC_BASE_URL']}")

In [ ]:
# ─── Cellule 3 : cleanup + cloudflared tunnel #2 (encode.jinkohub.digital → :8001) ───
import subprocess, time, socket
from google.colab import userdata

TOKEN_TUNNEL_2 = userdata.get("TOKEN_TUNNEL_2")
assert TOKEN_TUNNEL_2, "TOKEN_TUNNEL_2 manquant (Colab secrets, doit correspondre au tunnel #2 docfinder-query-encoder)."
assert "40899f6b" not in TOKEN_TUNNEL_2, "ERREUR : c'est le TOKEN du tunnel #1 (Mac), il faut celui du #2 (Colab)."

# Kill tout ce qui traîne d'un run précédent (uvicorn orphelin + cloudflared)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
subprocess.run(["fuser", "-k", "8001/tcp"], capture_output=True)
time.sleep(2)

# Vérif :8001 libre
s = socket.socket()
try:
    s.bind(("0.0.0.0", 8001))
    print("[ok] port 8001 libre")
finally:
    s.close()

# Install cloudflared si absent
if subprocess.run(["which", "cloudflared"], capture_output=True).returncode != 0:
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null

# Relance cloudflared en arrière-plan
subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", TOKEN_TUNNEL_2],
    stdout=open("/tmp/cloudflared.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(4)
print("[ok] cloudflared lancé — log : /tmp/cloudflared.log")
!tail -n 20 /tmp/cloudflared.log

In [ ]:
# ─── Cellule 4 : préflight CF Access (fail-fast) ───
# Si la policy CF Access n'autorise pas ce service token, on reçoit du HTML au lieu de JSON.
# On le détecte AVANT le pipeline pour éviter le `r.json()` silencieux dans list_files.
import os, httpx

url = f"{os.environ['MAC_BASE_URL']}/files/list"
headers = {
    "CF-Access-Client-Id":     os.environ["CF_ACCESS_CLIENT_ID"],
    "CF-Access-Client-Secret": os.environ["CF_ACCESS_CLIENT_SECRET"],
}
r = httpx.get(url, params={"root": os.environ["DOCFINDER_ROOT"], "limit": 1},
              headers=headers, follow_redirects=True, timeout=30)

ct = r.headers.get("content-type", "")
print(f"status        : {r.status_code}")
print(f"final url     : {r.url}")
print(f"content-type  : {ct}")

if "application/json" not in ct:
    print(f"\nbody (300 chars): {r.text[:300]!r}")
    raise SystemExit(
        "\n❌ CF Access n'accepte pas ce service token pour docfinder.jinkohub.digital.\n"
        "   → Dashboard Zero Trust → Access → Applications → docfinder → onglet Policies\n"
        "   → ajoute une policy 'Service Auth' avec Include > Service Token > <token Colab>."
    )

n = len(r.json())
print(f"\n✅ CF Access OK — /files/list renvoie {n} fichier(s) (test limit=1).")

In [ ]:
# ─── Cellule 5 : flags runtime ───
# Tourne-boutons pour le pipeline. À ajuster avant cellule 6.
import os

# Force la réindexation complète (bypass du check mtime).
# "1" pour valider le chemin GPU de bout en bout quand tout est déjà à jour.
# ""  (vide) pour le mode incrémental habituel.
os.environ["DOCFINDER_FORCE_REINDEX"] = "1"

# Limite le nombre de docs traités (smoke test). "" = pas de limite.
os.environ["DOCFINDER_MAX_DOCS"] = "20"

print(f"FORCE_REINDEX = {os.environ.get('DOCFINDER_FORCE_REINDEX') or '(désactivé)'}")
print(f"MAX_DOCS      = {os.environ.get('DOCFINDER_MAX_DOCS') or '(aucune limite)'}")

In [ ]:
# ─── Cellule 6 : query_server (:8001) + pipeline d'indexation ───
# Charge BGE-M3 une fois sur GPU, partage le wrapper entre uvicorn et run_pipeline.
exec(open("/content/docfinder/colab_helpers_cell.py").read())